# PII NER — training & TFLite export (Colab / GPU)

Fine-tunes `distilbert-base-uncased` for token-level PII detection and exports the
TFLite model + tokenizer files consumed by the PrivGuard Flutter app.

The model tags 17 PII entity types (names, addresses, phone / credit-card / tax /
ID numbers, email, date of birth, …) as 35 BIO labels. A GPU runtime is required —
training on CPU is impractically slow.

## App I/O contract — do not change

The Flutter app loads the exported model positionally, so these must stay fixed:

- **Inputs:** `input_ids[1, 128] int32`, `attention_mask[1, 128] int32`
- **Output:** `logits[1, 128, 35]`
- **Labels:** the 35 BIO labels, in the exact order defined in the label-set cell.

The final cells assert this contract before packaging the assets.

## How to run

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run **STEP 1** (install), then **Runtime → Restart session** — mandatory; see
   the note in that cell.
3. **Runtime → Run all**.
4. Optionally tune `SAMPLE_SIZE` / `EPOCHS` in the config cell to trade accuracy
   for runtime.
5. At the end, `pii_ner_assets.zip` downloads. Copy its 4 files —
   `pii_model.tflite`, `vocab.txt`, `tokenizer.json`, `tokenizer_config.json` —
   into the repo's **`assets/`** folder, overwriting the existing ones.

In [1]:
# =====================================================================
# STEP 1 — Install the EXACT pinned stack (GPU-enabled).
# Run this cell, then RESTART the runtime (Runtime > Restart session)
# BEFORE running anything below.
#
# WHY THE RESTART IS MANDATORY:
#   Colab preloads a newer numpy (2.x) at kernel startup. We downgrade to
#   numpy 1.26 (TF 2.17 requires numpy<2). A downgrade of an already-imported
#   C-extension package only takes effect in a FRESH session. The version-guard
#   cell (STEP 2) will hard-fail if the restart is skipped.
#
# WHY THESE PINS (mirror the project's requirements.txt — do NOT bump):
#   * transformers 5.x DROPPED every TF* model class -> must stay on 4.44.x.
#   * transformers' TF models need the Keras-2 API -> tf-keras + TF_USE_LEGACY_KERAS=1
#     (Colab ships Keras 3, which is incompatible; the env var routes around it).
#   * TF 2.17 requires numpy<2 and protobuf<5.
#   * tokenizers / huggingface_hub ranges are what transformers 4.44 needs.
#   * "tensorflow[and-cuda]" pulls TF's OWN matching CUDA/cuDNN wheels, so the
#     GPU is detected regardless of what CUDA Colab happens to ship. This is the
#     reproducible choice — it will not fight Colab's system libraries.
# =====================================================================
!pip install -q \
  "tensorflow[and-cuda]==2.17.0" \
  "tf-keras==2.17.0" \
  "transformers==4.44.2" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.24.6" \
  "safetensors==0.4.5" \
  "numpy==1.26.4" \
  "protobuf==4.25.4" \
  "scikit-learn==1.5.2"

print("\n\n" + "=" * 68)
print("Install finished.  NEXT:  Runtime > Restart session,  then  Run all.")
print("=" * 68)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0

In [2]:
!pip uninstall -y jax jaxlib flax

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: flax 0.11.2
Uninstalling flax-0.11.2:
  Successfully uninstalled flax-0.11.2


In [1]:
import os
os.environ["USE_TF"] = "1"      # force transformers onto the TF backend
os.environ["USE_TORCH"] = "0"   # <-- stop transformers importing Colab's broken torch
os.environ["USE_FLAX"] = "0"

In [2]:
# =====================================================================
# STEP 2 — Imports, environment flags, and version/GPU guards.
# TF_USE_LEGACY_KERAS MUST be set BEFORE tensorflow is imported.
# =====================================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # transformers TF models -> Keras 2 (tf-keras)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hush TF's info/warning spam

import re, glob, json, shutil, pickle
from pathlib import Path
import numpy as np
import tensorflow as tf
import transformers, tokenizers, huggingface_hub
from transformers import AutoTokenizer, TFDistilBertForTokenClassification
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, snapshot_download

# ---- Version guard: fails LOUDLY if the post-install restart was skipped ----
print("tensorflow  :", tf.__version__)
print("numpy       :", np.__version__)
print("transformers:", transformers.__version__)
print("tokenizers  :", tokenizers.__version__)
print("hf_hub      :", huggingface_hub.__version__)
assert tf.__version__.startswith("2.17"),     "TF is not 2.17 — re-run STEP 1 then RESTART."
assert np.__version__.startswith("1.26"),     "numpy is not 1.26 — the RESTART after STEP 1 was skipped."
assert transformers.__version__ == "4.44.2",  "transformers is not 4.44.2 — re-run STEP 1 then RESTART."

# ---- GPU guard: this notebook is impractical on CPU (hours vs minutes) ------
gpus = tf.config.list_physical_devices("GPU")
print("GPUs        :", gpus)
assert gpus, "No GPU visible! Runtime > Change runtime type > T4 GPU, then Run all."

tf.random.set_seed(42)
np.random.seed(42)
print("\nEnvironment OK — GPU ready.")

tensorflow  : 2.17.0
numpy       : 1.26.4
transformers: 4.44.2
tokenizers  : 0.19.1
hf_hub      : 0.24.6
GPUs        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Environment OK — GPU ready.


In [3]:
# =====================================================================
# STEP 3 — Paths.
# On Colab there is no git repo, so we work under /content and collect the
# 4 shippable files into ASSETS_DIR, which gets zipped & downloaded at the end.
#
# OPTIONAL persistence: set USE_DRIVE = True to write outputs to your Google
# Drive instead of ephemeral /content. Handy if the runtime disconnects — the
# trained checkpoint survives so you can re-run only the TFLite export. Default
# is False (simplest; everything is downloaded as a zip at the end anyway).
# =====================================================================
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/pii_ner")
else:
    BASE_DIR = Path("/content/pii_ner")

DATA_DIR   = BASE_DIR / "data"       # dataset cache
MODEL_DIR  = BASE_DIR / "pii_model"  # HF checkpoint + tokenizer
ASSETS_DIR = BASE_DIR / "assets"     # <-- the 4 files you copy into the app's assets/
for d in (DATA_DIR, MODEL_DIR, ASSETS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR  :", BASE_DIR)
print("DATA_DIR  :", DATA_DIR)
print("MODEL_DIR :", MODEL_DIR)
print("ASSETS_DIR:", ASSETS_DIR)

BASE_DIR  : /content/pii_ner
DATA_DIR  : /content/pii_ner/data
MODEL_DIR : /content/pii_ner/pii_model
ASSETS_DIR: /content/pii_ner/assets


In [4]:
# ---- Configuration -----------------------------------------------------------
# Only SAMPLE_SIZE and EPOCHS are meant to be tuned; the rest are fixed by the
# app I/O contract or are validated defaults. All are env-overridable.

# >>> CHANGE THIS ONE LINE to trade accuracy for runtime <<<
# 100k gives the best coverage of rare entity types (credit-card, tax, driver-
# licence) and takes ~70-130 min on a T4. If the Colab session disconnects before
# training finishes, drop it to 60000 (~40-80 min).
SAMPLE_SIZE    = int(os.environ.get("PII_SAMPLE_SIZE", 100_000))  # training examples

MAX_LENGTH     = 128            # MUST stay 128 — matches the Flutter app I/O
BATCH_SIZE     = 16             # a T4 also handles 32 (~2x faster); 16 is the validated default
EPOCHS         = int(os.environ.get("PII_EPOCHS", 6))  # ceiling; early-stopping trims the tail
LEARNING_RATE  = 2e-5
VAL_SPLIT      = 0.2
COMPARE_SUBSET = int(os.environ.get("PII_COMPARE_SUBSET", 400))  # #val examples for TFLite compare
print(f"SAMPLE_SIZE={SAMPLE_SIZE}  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}  "
      f"MAX_LENGTH={MAX_LENGTH}  COMPARE_SUBSET={COMPARE_SUBSET}")

SAMPLE_SIZE=100000  EPOCHS=6  BATCH_SIZE=16  MAX_LENGTH=128  COMPARE_SUBSET=400


In [5]:
# ---- Label set: EXACT same names AND order as the Flutter app (35 labels) --
# Do NOT reorder — the app maps output indices -> these labels positionally.
LABEL_NAMES = ['O', 'B-SURNAME', 'I-SURNAME', 'B-CITY', 'I-CITY', 'B-GIVENNAME', 'I-GIVENNAME',
               'B-DATEOFBIRTH', 'I-DATEOFBIRTH', 'B-DRIVERLICENSENUM', 'I-DRIVERLICENSENUM',
               'B-CREDITCARDNUMBER', 'I-CREDITCARDNUMBER', 'B-TELEPHONENUM', 'I-TELEPHONENUM',
               'B-SOCIALNUM', 'I-SOCIALNUM', 'B-ZIPCODE', 'I-ZIPCODE', 'B-IDCARDNUM', 'I-IDCARDNUM',
               'B-USERNAME', 'I-USERNAME', 'B-STREET', 'I-STREET', 'B-TAXNUM', 'I-TAXNUM',
               'B-EMAIL', 'I-EMAIL', 'B-BUILDINGNUM', 'I-BUILDINGNUM', 'B-PASSWORD', 'I-PASSWORD',
               'B-ACCOUNTNUM', 'I-ACCOUNTNUM']
LABEL_TO_ID = {l: i for i, l in enumerate(LABEL_NAMES)}
ID_TO_LABEL = {i: l for i, l in enumerate(LABEL_NAMES)}
NUM_LABELS  = len(LABEL_NAMES)
ENTITY_TYPES = {l[2:] for l in LABEL_NAMES if l != 'O'}
assert NUM_LABELS == 35
print(NUM_LABELS, "labels;", len(ENTITY_TYPES), "entity types")

35 labels; 17 entity types


In [6]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def download_english_train():
    """Download ONLY the English train shard(s) into DATA_DIR (keeps it small).
    Returns a list of local .jsonl file paths. Dataset is public — no HF login."""
    api = HfApi()
    files = api.list_repo_files("ai4privacy/pii-masking-400k", repo_type="dataset")
    jsonl = [f for f in files if f.endswith('.jsonl')]
    def is_en(f):
        bn = os.path.basename(f).lower()
        return bn.endswith('en.jsonl') or 'english' in bn   # e.g. '1en.jsonl'
    en_train = [f for f in jsonl if 'train' in f.lower() and is_en(f)]
    if not en_train:            # fallbacks if naming differs
        en_train = [f for f in jsonl if is_en(f)] or [f for f in jsonl if 'train' in f.lower()] or jsonl
    print("Fetching:", en_train)
    snap = snapshot_download("ai4privacy/pii-masking-400k", repo_type="dataset",
                             cache_dir=str(DATA_DIR), allow_patterns=en_train)
    return sorted(glob.glob(os.path.join(snap, "**", "*.jsonl"), recursive=True))

def load_records(paths, limit=None):
    """Load jsonl records (each has source_text + privacy_mask)."""
    records = []
    for fp in paths:
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                records.append(json.loads(line))
                if limit and len(records) >= limit:
                    return records
    return records

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
def encode_example(record):
    """Tokenize source_text ONCE and align BIO labels via char offsets.

    Special tokens ([CLS]/[SEP]) and padding get label -100 so they are ignored
    by the masked loss. Returns (input_ids, attention_mask, label_ids), each
    length MAX_LENGTH."""
    text = record.get("source_text", "") or ""
    spans = record.get("privacy_mask", []) or []

    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding="max_length",
                    return_offsets_mapping=True, return_special_tokens_mask=True)
    offsets = enc["offset_mapping"]
    special = enc["special_tokens_mask"]

    labels = []
    for (s, e), sp in zip(offsets, special):
        # special token or zero-width (padding) -> ignore in loss
        labels.append(-100 if (sp == 1 or s == e) else LABEL_TO_ID['O'])

    for ent in spans:
        etype = str(ent.get("label", "")).upper()
        if etype not in ENTITY_TYPES:
            continue
        es, ee = ent["start"], ent["end"]
        first = True
        for i, (s, e) in enumerate(offsets):
            if labels[i] == -100:
                continue
            if s < ee and e > es:                       # token overlaps entity span
                labels[i] = LABEL_TO_ID[f"B-{etype}" if first else f"I-{etype}"]
                first = False
    return enc["input_ids"], enc["attention_mask"], labels

def build_arrays(records):
    ii, am, lb = [], [], []
    for r in records:
        a, b, c = encode_example(r)
        ii.append(a); am.append(b); lb.append(c)
    return (np.asarray(ii, dtype=np.int32),
            np.asarray(am, dtype=np.int32),
            np.asarray(lb, dtype=np.int32))

In [8]:
# ---- Entity-level metrics (span match), no external deps -------------------
def extract_spans(tags):
    """BIO tag list -> set of (type, start, end) spans."""
    spans, cur = [], None
    for i, tag in enumerate(tags):
        if tag == 'O':
            if cur: spans.append(tuple(cur)); cur = None
            continue
        prefix, _, etype = tag.partition('-')
        if prefix == 'B':
            if cur: spans.append(tuple(cur))
            cur = [etype, i, i]
        else:  # 'I' — extend if same type, else treat leniently as a new span
            if cur and cur[0] == etype:
                cur[2] = i
            else:
                if cur: spans.append(tuple(cur))
                cur = [etype, i, i]
    if cur: spans.append(tuple(cur))
    return set(spans)

def entity_f1(true_id_rows, pred_id_rows):
    """Micro P/R/F1 over entity spans, plus a per-type breakdown.

    Positions with true == -100 are ignored. The returned dict carries a
    'per_type' map {type: {precision, recall, f1, tp, fp, fn, support}} so
    weak entity types can be spotted directly instead of inferred from spot
    tests."""
    tp = fp = fn = 0
    per = {}  # type -> [tp, fp, fn]
    def bump(typ, i):
        per.setdefault(typ, [0, 0, 0])[i] += 1
    for t_row, p_row in zip(true_id_rows, pred_id_rows):
        t_tags, p_tags = [], []
        for t, p in zip(t_row, p_row):
            if int(t) == -100:
                continue
            t_tags.append(ID_TO_LABEL[int(t)])
            p_tags.append(ID_TO_LABEL[int(p)])
        ts, ps = extract_spans(t_tags), extract_spans(p_tags)
        tp += len(ts & ps); fp += len(ps - ts); fn += len(ts - ps)
        for span in (ts & ps): bump(span[0], 0)   # tp (matched type)
        for span in (ps - ts): bump(span[0], 1)   # fp (by predicted type)
        for span in (ts - ps): bump(span[0], 2)   # fn (by true type)
    def prf(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        return p, r, f
    per_type = {}
    for typ, (t, fp_c, fn_c) in per.items():
        p, r, f = prf(t, fp_c, fn_c)
        per_type[typ] = {"precision": p, "recall": r, "f1": f,
                         "tp": t, "fp": fp_c, "fn": fn_c, "support": t + fn_c}
    p, r, f = prf(tp, fp, fn)
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn,
            "per_type": per_type}

def print_entity_report(true_id_rows, pred_id_rows, per_type=True):
    m = entity_f1(true_id_rows, pred_id_rows)
    print(f"Entity micro  P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1']:.4f}  "
          f"(tp={m['tp']} fp={m['fp']} fn={m['fn']})")
    if per_type:
        print(f"\n{'entity':<18}{'P':>7}{'R':>7}{'F1':>7}{'support':>9}")
        print("-" * 48)
        # weakest F1 first, so the problem types are impossible to miss
        for typ, d in sorted(m['per_type'].items(), key=lambda kv: kv[1]['f1']):
            print(f"{typ:<18}{d['precision']:>7.3f}{d['recall']:>7.3f}"
                  f"{d['f1']:>7.3f}{d['support']:>9}")
    return m

In [9]:
paths = download_english_train()
records = load_records(paths, limit=SAMPLE_SIZE)
print(f"Loaded {len(records)} records")
print("Sample keys:", list(records[0].keys()) if records else "none")

# sanity: show label alignment on one example
if records:
    ii0, am0, lb0 = encode_example(records[0])
    toks = tokenizer.convert_ids_to_tokens(ii0)
    shown = [(t, ID_TO_LABEL[l]) for t, l in zip(toks, lb0) if l != -100][:25]
    print("source_text:", records[0]['source_text'][:120])
    print("aligned (token,label):", shown)

Fetching: ['data/train/1en.jsonl']


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

1en.jsonl:   0%|          | 0.00/84.8M [00:00<?, ?B/s]

Loaded 68275 records
Sample keys: ['source_text', 'locale', 'language', 'split', 'privacy_mask', 'uid', 'masked_text', 'mbert_tokens', 'mbert_token_classes']
source_text: <p>My child faozzsd379223 (DOB: May/58) will undergo treatment with Dr. faozzsd379223, office at Hill Road. Our ZIP code
aligned (token,label): [('<', 'O'), ('p', 'O'), ('>', 'O'), ('my', 'O'), ('child', 'O'), ('fa', 'B-USERNAME'), ('##oz', 'I-USERNAME'), ('##z', 'I-USERNAME'), ('##sd', 'I-USERNAME'), ('##37', 'I-USERNAME'), ('##9', 'I-USERNAME'), ('##22', 'I-USERNAME'), ('##3', 'I-USERNAME'), ('(', 'O'), ('do', 'O'), ('##b', 'O'), (':', 'O'), ('may', 'B-DATEOFBIRTH'), ('/', 'I-DATEOFBIRTH'), ('58', 'I-DATEOFBIRTH'), (')', 'O'), ('will', 'O'), ('undergo', 'O'), ('treatment', 'O'), ('with', 'O')]


In [10]:
input_ids, attention_masks, label_ids = build_arrays(records)
print("input_ids:", input_ids.shape, "attention_masks:", attention_masks.shape, "labels:", label_ids.shape)
# label distribution sanity (excluding ignored -100)
uniq, cnt = np.unique(label_ids[label_ids != -100], return_counts=True)
nonO = {ID_TO_LABEL[int(u)]: int(c) for u, c in zip(uniq, cnt) if int(u) != 0}
print("non-O label counts:", dict(sorted(nonO.items(), key=lambda x: -x[1])))

input_ids: (68275, 128) attention_masks: (68275, 128) labels: (68275, 128)
non-O label counts: {'I-EMAIL': 56545, 'I-TELEPHONENUM': 38233, 'I-USERNAME': 36245, 'I-ACCOUNTNUM': 29782, 'I-IDCARDNUM': 22200, 'I-GIVENNAME': 18055, 'I-SURNAME': 16471, 'I-SOCIALNUM': 15883, 'I-PASSWORD': 15815, 'I-DRIVERLICENSENUM': 15503, 'I-DATEOFBIRTH': 14449, 'I-CREDITCARDNUMBER': 14278, 'I-CITY': 12341, 'I-TAXNUM': 12017, 'B-GIVENNAME': 11875, 'I-ZIPCODE': 10550, 'B-SURNAME': 8614, 'B-CITY': 7780, 'B-USERNAME': 7372, 'I-STREET': 7162, 'B-EMAIL': 6556, 'B-TELEPHONENUM': 5094, 'B-IDCARDNUM': 4053, 'B-BUILDINGNUM': 3917, 'B-ACCOUNTNUM': 3765, 'B-ZIPCODE': 3733, 'B-DATEOFBIRTH': 3477, 'B-STREET': 3349, 'B-SOCIALNUM': 2886, 'B-PASSWORD': 2634, 'I-BUILDINGNUM': 2512, 'B-TAXNUM': 2237, 'B-DRIVERLICENSENUM': 2090, 'B-CREDITCARDNUMBER': 1660}


In [11]:
train_ii, val_ii, train_am, val_am, train_lb, val_lb = train_test_split(
    input_ids, attention_masks, label_ids, test_size=VAL_SPLIT, random_state=42)
print("train:", train_ii.shape[0], " val:", val_ii.shape[0])

train: 54620  val: 13655


In [12]:
# ---- Class weights: counter the heavy 'O' / rare-entity imbalance ----------
# The dataset is dominated by 'O' and by common types (names, cities), while
# credit-card / tax / driver-licence spans are scarce — which is what drives the
# rare-class confusions (e.g. credit-card tokens predicted as phone). We weight
# the per-token loss by capped inverse-sqrt frequency, computed on the TRAIN
# split only: 'O' is pinned to 1.0 and rare types get up to CLASS_WEIGHT_CAP x
# more loss. Raise/lower the cap to push harder/softer on the rare classes.
CLASS_WEIGHT_CAP = 20.0
_counts = np.bincount(train_lb[train_lb != -100].ravel(), minlength=NUM_LABELS).astype(np.float64)
_counts[_counts == 0] = 1.0
_freq = _counts / _counts.sum()
_w = 1.0 / np.sqrt(_freq)
_w = _w / _w[LABEL_TO_ID['O']]                 # normalize so 'O' == 1.0
_w = np.clip(_w, 1.0, CLASS_WEIGHT_CAP)
CLASS_WEIGHTS = tf.constant(_w, dtype=tf.float32)
print(f"class weights: min={_w.min():.2f} max={_w.max():.2f} (cap={CLASS_WEIGHT_CAP})")
for typ in ("CREDITCARDNUMBER", "TAXNUM", "DRIVERLICENSENUM", "TELEPHONENUM"):
    print(f"  B-{typ:<18} weight={_w[LABEL_TO_ID['B-'+typ]]:.2f}")

# ---- Masked, class-weighted loss/metric: ignore -100 (padding + specials) ---
def masked_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.not_equal(y_true, -100)
    y_safe = tf.where(mask, y_true, tf.zeros_like(y_true))
    per_tok = tf.keras.losses.sparse_categorical_crossentropy(y_safe, y_pred, from_logits=True)
    w = tf.gather(CLASS_WEIGHTS, y_safe)                 # per-token weight by true class
    mask_f = tf.cast(mask, per_tok.dtype)
    weighted = per_tok * mask_f * tf.cast(w, per_tok.dtype)
    # normalize by the (unweighted) token count -> keeps the loss magnitude stable
    return tf.reduce_sum(weighted) / tf.maximum(tf.reduce_sum(mask_f), 1.0)

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.not_equal(y_true, -100)
    preds = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
    hits = tf.logical_and(mask, tf.equal(preds, tf.where(mask, y_true, tf.zeros_like(y_true))))
    return tf.reduce_sum(tf.cast(hits, tf.float32)) / tf.maximum(tf.reduce_sum(tf.cast(mask, tf.float32)), 1.0)

def build_model():
    backbone = TFDistilBertForTokenClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=NUM_LABELS,
        id2label=ID_TO_LABEL, label2id=LABEL_TO_ID)
    ids  = tf.keras.Input(shape=(MAX_LENGTH,), dtype=tf.int32, name='input_ids')
    mask = tf.keras.Input(shape=(MAX_LENGTH,), dtype=tf.int32, name='attention_mask')
    logits = backbone(input_ids=ids, attention_mask=mask).logits
    model = tf.keras.Model([ids, mask], logits)
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss=masked_loss, metrics=[masked_accuracy])
    return model, backbone

class EntityF1Callback(tf.keras.callbacks.Callback):
    """Computes val entity-F1 each epoch -> logs['val_f1'] (for EarlyStopping)."""
    def __init__(self, vi, vm, vl):
        super().__init__(); self.vi, self.vm, self.vl = vi, vm, vl
    def on_epoch_end(self, epoch, logs=None):
        logits = self.model.predict({'input_ids': self.vi, 'attention_mask': self.vm},
                                    batch_size=BATCH_SIZE, verbose=0)
        preds = np.asarray(logits).argmax(-1)
        m = entity_f1(self.vl, preds)
        logs = logs or {}
        logs['val_f1'] = m['f1']
        print(f"  val entity-F1={m['f1']:.4f} (P={m['precision']:.3f} R={m['recall']:.3f})")

class weights: min=1.00 max=20.00 (cap=20.0)
  B-CREDITCARDNUMBER   weight=20.00
  B-TAXNUM             weight=20.00
  B-DRIVERLICENSENUM   weight=20.00
  B-TELEPHONENUM       weight=20.00


In [13]:
# ---- Train. On a T4 GPU this runs in minutes rather than hours. -------------
# The first PyTorch->TF weight load prints a few "weights not used / newly
# initialized" messages — expected (a fresh classifier head is added on top).
# EarlyStopping monitors val entity-F1 and restores the best epoch's weights, so
# EPOCHS is only a ceiling; patience=3 leaves room for a dip after an LR drop.
model, backbone = build_model()

callbacks = [
    EntityF1Callback(val_ii, val_am, val_lb),
    tf.keras.callbacks.EarlyStopping(monitor='val_f1', mode='max', patience=3,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=1, factor=0.5),
]

history = model.fit(
    {'input_ids': train_ii, 'attention_mask': train_am}, train_lb,
    validation_data=({'input_ids': val_ii, 'attention_mask': val_am}, val_lb),
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForTokenClassification: ['vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertForTokenClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForTokenClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForTokenClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able t

Epoch 1/6
3414/3414 [==============================] - 931s 266ms/step - loss: 1.0439 - masked_accuracy: 0.9384 - val_loss: 0.5778 - val_masked_accuracy: 0.9561 - val_f1: 0.7252 - lr: 2.0000e-05
Epoch 2/6
3414/3414 [==============================] - 913s 267ms/step - loss: 0.4268 - masked_accuracy: 0.9643 - val_loss: 0.4871 - val_masked_accuracy: 0.9611 - val_f1: 0.7627 - lr: 2.0000e-05
Epoch 3/6
3414/3414 [==============================] - 911s 267ms/step - loss: 0.3027 - masked_accuracy: 0.9716 - val_loss: 0.4750 - val_masked_accuracy: 0.9664 - val_f1: 0.7858 - lr: 2.0000e-05
Epoch 4/6
3414/3414 [==============================] - 911s 267ms/step - loss: 0.2149 - masked_accuracy: 0.9777 - val_loss: 0.4631 - val_masked_accuracy: 0.9702 - val_f1: 0.8020 - lr: 2.0000e-05
Epoch 5/6
3414/3414 [==============================] - 909s 266ms/step - loss: 0.1543 - masked_accuracy: 0.9823 - val_loss: 0.5132 - val_masked_accuracy: 0.9733 - val_f1: 0.8223 - lr: 2.0000e-05
Epoch 6/6
3414/3414 [====

In [14]:
# Final validation report: entity micro P/R/F1 + a per-type table (weakest first),
# so it's obvious which of the 17 entity types still need work.
val_logits = model.predict({'input_ids': val_ii, 'attention_mask': val_am},
                           batch_size=BATCH_SIZE, verbose=0)
val_pred = np.asarray(val_logits).argmax(-1)
print("Validation:")
_ = print_entity_report(val_lb, val_pred)

Validation:
Entity micro  P=0.7869  R=0.9026  F1=0.8408  (tp=14536 fp=3937 fn=1568)

entity                  P      R     F1  support
------------------------------------------------
ACCOUNTNUM          0.511  0.818  0.629      776
SOCIALNUM           0.627  0.859  0.725      582
DATEOFBIRTH         0.600  0.930  0.729      682
TAXNUM              0.661  0.815  0.730      405
SURNAME             0.749  0.810  0.779     1741
BUILDINGNUM         0.670  0.945  0.784      816
IDCARDNUM           0.711  0.894  0.792      799
CREDITCARDNUMBER    0.738  0.885  0.805      321
GIVENNAME           0.817  0.845  0.831     2286
DRIVERLICENSENUM    0.800  0.902  0.848      400
USERNAME            0.824  0.880  0.851     1480
ZIPCODE             0.846  0.965  0.902      742
PASSWORD            0.899  0.936  0.918      487
STREET              0.905  0.957  0.931      700
CITY                0.935  0.965  0.950     1584
TELEPHONENUM        0.965  0.989  0.977      997
EMAIL               0.998  0.999 

In [15]:
def predict_sentence(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding='max_length',
                    return_tensors='np')
    logits = model.predict({'input_ids': enc['input_ids'].astype(np.int32),
                            'attention_mask': enc['attention_mask'].astype(np.int32)}, verbose=0)
    ids = np.asarray(logits)[0].argmax(-1)
    toks = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    ents, cur = [], None
    for tok, lid in zip(toks, ids):
        if tok in ('[CLS]', '[SEP]', '[PAD]'):
            continue
        lab = ID_TO_LABEL[int(lid)]
        if lab.startswith('B-'):
            if cur: ents.append(cur)
            cur = {'text': tok.replace('##', ''), 'label': lab[2:]}
        elif lab.startswith('I-') and cur:
            cur['text'] += tok.replace('##', '')
        else:
            if cur: ents.append(cur); cur = None
    if cur: ents.append(cur)
    return ents

for s in ["My name is John Smith and my email is john.smith@email.com",
          "Please call me at 555-123-4567 or visit me at 123 Main Street",
          "My credit card number is 4532-1234-5678-9012"]:
    print(s)
    for e in predict_sentence(s):
        print("   -", e['text'], "(", e['label'], ")")

My name is John Smith and my email is john.smith@email.com
   - johnsmith ( GIVENNAME )
   - john.smith@email.com ( EMAIL )
Please call me at 555-123-4567 or visit me at 123 Main Street
   - 555-123-4567 ( TELEPHONENUM )
   - 123 ( BUILDINGNUM )
   - mainstreet ( STREET )
My credit card number is 4532-1234-5678-9012
   - 4532-1234-5678-9012 ( TELEPHONENUM )


In [16]:
# Save HF checkpoint + tokenizer + label mappings into MODEL_DIR.
backbone.save_pretrained(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
with open(MODEL_DIR / 'label_mappings.pkl', 'wb') as f:
    pickle.dump({'label_to_id': LABEL_TO_ID, 'id_to_label': ID_TO_LABEL, 'label_names': LABEL_NAMES}, f)
print("Saved checkpoint to", MODEL_DIR)

Saved checkpoint to /content/pii_ner/pii_model


## TFLite export — convert, compare, ship the winner

Convert the trained model to two candidates — float32 (no quantization) and
dynamic-range int8 weights — score each on a held-out subset with entity-F1,
then ship the winner as `pii_model.tflite`. The app is mobile, so the ~4x smaller
int8 model is strongly preferred and wins unless float32 is materially more
accurate. The I/O signature is unchanged:
`input_ids[1,128] int32 + attention_mask[1,128] int32 -> logits[1,128,35]`.

In [17]:
def _concrete_func(model):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[1, MAX_LENGTH], dtype=tf.int32, name='input_ids'),
        tf.TensorSpec(shape=[1, MAX_LENGTH], dtype=tf.int32, name='attention_mask')])
    def f(input_ids, attention_mask):
        return model(input_ids=input_ids, attention_mask=attention_mask, training=False).logits
    return f.get_concrete_function()

def convert_tflite(hf_model, dynamic_range: bool, out_path: Path):
    conv = tf.lite.TFLiteConverter.from_concrete_functions([_concrete_func(hf_model)],
                                                           trackable_obj=hf_model)
    if dynamic_range:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]   # int8 weights only
    tfl = conv.convert()
    out_path.write_bytes(tfl)
    return len(tfl) / 1024 / 1024  # MB

def eval_tflite_f1(tflite_path, vi, vm, vl, n):
    interp = tf.lite.Interpreter(model_path=str(tflite_path)); interp.allocate_tensors()
    inp = interp.get_input_details(); out = interp.get_output_details()
    idx = {'input_ids': None, 'attention_mask': None}
    for d in inp:
        for k in idx:
            if k in d['name']:
                idx[k] = d['index']
    if idx['input_ids'] is None:   # fallback by order
        idx['input_ids'], idx['attention_mask'] = inp[1]['index'], inp[0]['index']
    preds = []
    for i in range(min(n, vi.shape[0])):
        interp.set_tensor(idx['input_ids'], vi[i:i+1].astype(np.int32))
        interp.set_tensor(idx['attention_mask'], vm[i:i+1].astype(np.int32))
        interp.invoke()
        preds.append(interp.get_tensor(out[0]['index'])[0].argmax(-1))
    preds = np.asarray(preds)
    return entity_f1(vl[:preds.shape[0]], preds), out[0]['shape']

In [18]:
hf = TFDistilBertForTokenClassification.from_pretrained(str(MODEL_DIR))

cand_dir = BASE_DIR / "_tflite_candidates"; cand_dir.mkdir(exist_ok=True)
no_quant_p = cand_dir / "pii_model_no_quant.tflite"
dynamic_p  = cand_dir / "pii_model_dynamic.tflite"

print("Converting candidates...")
size_nq = convert_tflite(hf, dynamic_range=False, out_path=no_quant_p)
size_dy = convert_tflite(hf, dynamic_range=True,  out_path=dynamic_p)

print(f"Scoring on {COMPARE_SUBSET} val examples...")
f1_nq, shape_nq = eval_tflite_f1(no_quant_p, val_ii, val_am, val_lb, COMPARE_SUBSET)
f1_dy, shape_dy = eval_tflite_f1(dynamic_p,  val_ii, val_am, val_lb, COMPARE_SUBSET)

print("\n%-14s %8s %10s  %s" % ("candidate", "size(MB)", "entityF1", "out_shape"))
print("%-14s %8.1f %10.4f  %s" % ("no_quant", size_nq, f1_nq['f1'], list(shape_nq)))
print("%-14s %8.1f %10.4f  %s" % ("dynamic",  size_dy, f1_dy['f1'], list(shape_dy)))

# Ship decision: this is a mobile model, so the ~4x smaller int8 build wins
# unless float32 is more than 1.0 F1-point better (float32 is ~4x the size and
# usually gains almost nothing over dynamic-range quantization here).
MOBILE_F1_MARGIN = 0.01
if f1_dy['f1'] >= f1_nq['f1'] - MOBILE_F1_MARGIN:
    winner, wsize, wf1 = dynamic_p, size_dy, f1_dy['f1']
    print(f"\nWINNER: dynamic-range int8 ({wsize:.1f} MB, F1={wf1:.4f}) — much smaller, F1 within margin")
else:
    winner, wsize, wf1 = no_quant_p, size_nq, f1_nq['f1']
    print(f"\nWINNER: no-quant float32 ({wsize:.1f} MB, F1={wf1:.4f}) — F1 gain exceeds the mobile size margin")
print("(the next cell ships whichever candidate won above)")

Some layers from the model checkpoint at /content/pii_ner/pii_model were not used when initializing TFDistilBertForTokenClassification: ['dropout_19']
- This IS expected if you are initializing TFDistilBertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForTokenClassification were not initialized from the model checkpoint at /content/pii_ner/pii_model and are newly initialized: ['dropout_39']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Converting candidates...


Scoring on 400 val examples...

candidate      size(MB)   entityF1  out_shape
no_quant          252.2     0.8473  [1, 128, 35]
dynamic            64.0     0.8440  [1, 128, 35]

WINNER: dynamic-range int8 (64.0 MB, F1=0.8440) — much smaller, F1 within margin
(the next cell ships whichever candidate won above)


In [19]:
# Ship the winner chosen above under the filename the app loads; sync tokenizer files too.
target = ASSETS_DIR / "pii_model.tflite"
shutil.copyfile(winner, target)
for fn in ("vocab.txt", "tokenizer.json", "tokenizer_config.json"):
    src = MODEL_DIR / fn
    if src.exists():
        shutil.copyfile(src, ASSETS_DIR / fn)
print("Exported:", target, f"({target.stat().st_size/1024/1024:.1f} MB, F1={wf1:.4f})")
print("Collected files in", ASSETS_DIR, ":", [p.name for p in ASSETS_DIR.iterdir()])

Exported: /content/pii_ner/assets/pii_model.tflite (64.0 MB, F1=0.8440)
Collected files in /content/pii_ner/assets : ['tokenizer_config.json', 'vocab.txt', 'pii_model.tflite', 'tokenizer.json']


In [20]:
# Verify the shipped model matches the app contract: [1,128,35] logits.
interp = tf.lite.Interpreter(model_path=str(ASSETS_DIR / 'pii_model.tflite'))
interp.allocate_tensors()
outd = interp.get_output_details()[0]
print("Output shape:", list(outd['shape']), "(expected [1, 128, 35])")
assert list(outd['shape']) == [1, MAX_LENGTH, NUM_LABELS], "Output signature changed — app would break!"
print("OK — model I/O matches the Flutter app contract.")

Output shape: [1, 128, 35] (expected [1, 128, 35])
OK — model I/O matches the Flutter app contract.


In [22]:
# =====================================================================
# Figures for the README / report. Run AFTER training + the TFLite compare cell
# (needs: history, val_lb, val_pred, label_ids, f1_nq/f1_dy, size_nq/size_dy).
# Saves 5 PNGs into ./images and downloads images.zip.
# =====================================================================
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

IMAGES_DIR = Path("images"); IMAGES_DIR.mkdir(exist_ok=True)

# ---- Validated light-mode palette (colorblind-safe) -------------------------
SURFACE, INK, SECOND, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASE = "#e1e0d9", "#c3c2b7"
BLUE, AQUA, ORANGE = "#2a78d6", "#1baf7a", "#eb6834"
GRAY_BAR = "#c9c8c2"   # "before"/secondary bar

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "text.color": INK, "axes.labelcolor": SECOND, "axes.edgecolor": BASE,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlecolor": INK,
})

def _style(ax, grid_axis="y"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(BASE)
    ax.grid(True, axis=grid_axis, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    return ax

def _save(fig, name):
    p = IMAGES_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig); print("saved", p)

# ---- Derived numbers --------------------------------------------------------
m = entity_f1(val_lb, val_pred)                       # micro + per_type
per = m["per_type"]
n_records = len(records)
n_train, n_val = int(train_ii.shape[0]), int(val_ii.shape[0])
final_f1 = m["f1"]

# =============================== 1. Training curves ==========================
h = history.history
epochs = np.arange(1, len(h["val_f1"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
_style(a1)
a1.plot(epochs, h["val_f1"], color=BLUE, lw=2, marker="o", ms=6, zorder=3)
a1.annotate(f"{h['val_f1'][-1]:.3f}", (epochs[-1], h["val_f1"][-1]),
            textcoords="offset points", xytext=(-4, 8), ha="right",
            color=BLUE, fontweight="bold")
a1.set_title("Validation entity-F1 by epoch"); a1.set_xlabel("epoch"); a1.set_ylabel("entity-F1")
a1.set_xticks(epochs)
_style(a2)
a2.plot(epochs, h["loss"], color=BLUE, lw=2, marker="o", ms=5, label="train loss", zorder=3)
a2.plot(epochs, h["val_loss"], color=ORANGE, lw=2, marker="s", ms=5, label="val loss", zorder=3)
a2.set_title("Training vs validation loss"); a2.set_xlabel("epoch"); a2.set_ylabel("loss")
a2.set_xticks(epochs); a2.legend(frameon=False, labelcolor=SECOND)
fig.tight_layout(); _save(fig, "training_curves.png")

# =========================== 2. Before/after compression =====================
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
labels = ["float32\n(no quant)", "int8\n(dynamic)"]
colors = [GRAY_BAR, BLUE]
_style(a1)
b = a1.bar(labels, [f1_nq["f1"], f1_dy["f1"]], color=colors, width=0.6, zorder=3)
for rect, v in zip(b, [f1_nq["f1"], f1_dy["f1"]]):
    a1.text(rect.get_x() + rect.get_width()/2, v + 0.012, f"{v:.4f}",
            ha="center", color=INK, fontweight="bold")
a1.set_ylim(0, 1.0); a1.set_ylabel("entity-F1")
a1.set_title(f"Accuracy: Δ F1 = {f1_dy['f1'] - f1_nq['f1']:+.4f}")
_style(a2)
b = a2.bar(labels, [size_nq, size_dy], color=colors, width=0.6, zorder=3)
for rect, v in zip(b, [size_nq, size_dy]):
    a2.text(rect.get_x() + rect.get_width()/2, v + size_nq*0.02, f"{v:.1f} MB",
            ha="center", color=INK, fontweight="bold")
a2.set_ylabel("model size (MB)")
a2.set_title(f"Size: {size_nq/size_dy:.1f}× smaller ({(size_dy/size_nq - 1)*100:.0f}%)")
fig.suptitle("Dynamic-range int8 quantization: ~4× smaller, F1 essentially unchanged",
             color=SECOND, fontsize=11)
fig.tight_layout(); _save(fig, "compression_comparison.png")

# =============================== 3. Per-entity F1 ============================
items = sorted(per.items(), key=lambda kv: kv[1]["f1"])   # weakest first
names = [k for k, _ in items]; f1s = [d["f1"] for _, d in items]; sup = [d["support"] for _, d in items]
fig, ax = plt.subplots(figsize=(8, 7)); _style(ax, grid_axis="x")
y = np.arange(len(names))
ax.barh(y, f1s, color=BLUE, height=0.72, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(names)
ax.set_xlim(0, 1.05); ax.set_xlabel("entity-F1")
for yi, v, s in zip(y, f1s, sup):
    ax.text(v + 0.012, yi, f"{v:.3f}", va="center", color=INK, fontsize=9, fontweight="bold")
    ax.text(0.008, yi, f"n={s}", va="center", color=MUTED, fontsize=8)
ax.set_title(f"Per-entity validation F1   ·   micro-F1 = {final_f1:.3f}")
fig.tight_layout(); _save(fig, "per_entity_f1.png")

# =========================== 4. Dataset composition =========================
lab = label_ids[label_ids != -100]
counts = np.bincount(lab, minlength=NUM_LABELS)
ent_counts = {ln[2:]: int(counts[LABEL_TO_ID[ln]]) for ln in LABEL_NAMES if ln.startswith("B-")}
ent_counts = dict(sorted(ent_counts.items(), key=lambda kv: kv[1]))   # ascending
fig, ax = plt.subplots(figsize=(8, 7)); _style(ax, grid_axis="x")
y = np.arange(len(ent_counts)); vals = list(ent_counts.values())
ax.barh(y, vals, color=AQUA, height=0.72, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(list(ent_counts.keys()))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
for yi, v in zip(y, vals):
    ax.text(v + max(vals)*0.01, yi, f"{v:,}", va="center", color=INK, fontsize=9)
ax.set_xlabel("entity instances (B- spans)")
ax.set_title(f"Training-data composition   ·   {n_records:,} records, {len(ent_counts)} PII types")
fig.text(0.5, -0.02, "Heavy class imbalance (rarest ~%dx below the most common) motivates the "
         "class-weighted loss." % round(max(vals)/max(min(vals), 1)),
         ha="center", color=SECOND, fontsize=10)
fig.tight_layout(); _save(fig, "dataset_distribution.png")

# =============================== 5. Summary card ============================
tiles = [("Records", f"{n_records:,}", BLUE),
         ("PII entity types", f"{len(per)}", AQUA),
         ("Validation F1", f"{final_f1:.3f}", "#0ca30c"),
         ("On-device size", f"{size_dy:.0f} MB", ORANGE)]
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6))
for ax, (label, value, c) in zip(axes, tiles):
    ax.axis("off")
    ax.text(0.5, 0.62, value, ha="center", va="center", fontsize=30, fontweight="bold", color=c)
    ax.text(0.5, 0.22, label, ha="center", va="center", fontsize=12, color=SECOND)
fig.suptitle("PrivGuard PII NER — DistilBERT token classifier (int8 TFLite, on-device)",
             color=INK, fontsize=13, fontweight="bold")
fig.tight_layout(); _save(fig, "summary_card.png")

# ---- Zip + download ---------------------------------------------------------
import shutil, os
shutil.make_archive("images", "zip", IMAGES_DIR)
print("\nZipped:", os.path.abspath("images.zip"))
try:
    from google.colab import files
    files.download("images.zip")
except Exception as e:
    print("Auto-download failed - grab images.zip from the Files panel.\nReason:", e)


saved images/training_curves.png
saved images/compression_comparison.png
saved images/per_entity_f1.png
saved images/dataset_distribution.png
saved images/summary_card.png

Zipped: /content/images.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Download the assets

The next cell zips the 4 shippable files and downloads `pii_ner_assets.zip`.
Unzip it and copy the files into the repo's **`assets/`** folder, overwriting the
existing ones:

- `pii_model.tflite`
- `vocab.txt`
- `tokenizer.json`
- `tokenizer_config.json`

If the browser blocks the auto-download, open the Files panel on the left and
download `/content/pii_ner_assets.zip` manually.

In [21]:
# Zip the assets and trigger a browser download.
zip_base = "/content/pii_ner_assets"
if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")
shutil.make_archive(zip_base, 'zip', ASSETS_DIR)
print("Zipped:", zip_base + ".zip",
      f"({os.path.getsize(zip_base + '.zip')/1024/1024:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("Auto-download failed — grab it from the Files panel at",
          zip_base + ".zip", "\nReason:", e)

Zipped: /content/pii_ner_assets.zip (51.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>